In [ ]:
import os
import random
import time
from pathlib import Path
from tqdm import tqdm
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn
import torchvision.transforms as T
from torchvision.transforms import functional as F

# Проверка CUDA
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def find_image_for_stem(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def dice_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7, threshold: float = 0.5) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    denom = preds.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean().item()

def iou_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7, threshold: float = 0.5) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection

    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

# =========================
# КОМБИНИРОВАННАЯ ФУНКЦИЯ ПОТЕРЬ
# =========================

class CombinedLoss(nn.Module):
    """
    Комбинированная функция потерь: Dice + BCE + Focal
    """
    def __init__(self, dice_weight=0.5, bce_weight=0.3, focal_weight=0.2):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.bce = nn.BCEWithLogitsLoss()
        self.focal = smp.losses.FocalLoss(mode='binary', alpha=0.75, gamma=2.0)
        
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.focal_weight = focal_weight
        
    def forward(self, logits, targets):
        dice_loss = self.dice(logits, targets)
        bce_loss = self.bce(logits, targets)
        focal_loss = self.focal(logits, targets)
        
        total_loss = (self.dice_weight * dice_loss + 
                      self.bce_weight * bce_loss + 
                      self.focal_weight * focal_loss)
        
        return total_loss

# =========================
# АУГМЕНТАЦИИ ЧЕРЕЗ TORCHVISION
# =========================

class TrainTransform:
    """Трансформации для тренировочного датасета"""
    def __init__(self, img_size=512):
        self.img_size = img_size
        
    def __call__(self, image, mask):
        # Конвертируем в тензоры
        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image).float() / 255.0
            mask = torch.from_numpy(mask).float()
        
        # Переставляем размерности для torchvision (C, H, W)
        if image.dim() == 3 and image.shape[-1] == 3:
            image = image.permute(2, 0, 1)
        if mask.dim() == 3 and mask.shape[-1] == 1:
            mask = mask.permute(2, 0, 1)
        elif mask.dim() == 2:
            mask = mask.unsqueeze(0)
        
        # Ресайз
        image = F.resize(image, (self.img_size, self.img_size))
        mask = F.resize(mask, (self.img_size, self.img_size))
        
        # Случайный горизонтальный флип
        if random.random() > 0.5:
            image = F.hflip(image)
            mask = F.hflip(mask)
        
        # Случайный вертикальный флип
        if random.random() > 0.7:
            image = F.vflip(image)
            mask = F.vflip(mask)
        
        # Случайный поворот
        if random.random() > 0.5:
            angle = random.uniform(-30, 30)
            image = F.rotate(image, angle, fill=0)
            mask = F.rotate(mask, angle, fill=0)
        
        # Случайное изменение яркости и контраста
        if random.random() > 0.5:
            brightness = random.uniform(0.8, 1.2)
            contrast = random.uniform(0.8, 1.2)
            image = F.adjust_brightness(image, brightness)
            image = F.adjust_contrast(image, contrast)
        
        # Добавляем случайный шум
        if random.random() > 0.7:
            noise = torch.randn_like(image) * 0.05
            image = image + noise
            image = torch.clamp(image, 0, 1)
        
        # Нормализация для EfficientNet
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std
        
        return image, mask

class ValTransform:
    """Трансформации для валидационного датасета (только ресайз и нормализация)"""
    def __init__(self, img_size=512):
        self.img_size = img_size
        
    def __call__(self, image, mask):
        # Конвертируем в тензоры
        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image).float() / 255.0
            mask = torch.from_numpy(mask).float()
        
        # Переставляем размерности
        if image.dim() == 3 and image.shape[-1] == 3:
            image = image.permute(2, 0, 1)
        if mask.dim() == 3 and mask.shape[-1] == 1:
            mask = mask.permute(2, 0, 1)
        elif mask.dim() == 2:
            mask = mask.unsqueeze(0)
        
        # Ресайз
        image = F.resize(image, (self.img_size, self.img_size))
        mask = F.resize(mask, (self.img_size, self.img_size))
        
        # Нормализация
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std
        
        return image, mask

# =========================
# DATASET С ПОДДЕРЖКОЙ АУГМЕНТАЦИЙ
# =========================

class BinarySegDataset(Dataset):
    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        img_size: int = 512,
        phase: str = 'train',  # 'train' или 'val'
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size
        self.phase = phase
        
        # Выбираем трансформации в зависимости от фазы
        if phase == 'train':
            self.transform = TrainTransform(img_size)
        else:
            self.transform = ValTransform(img_size)
        
        print(f"Loading dataset from {self.images_dir} and {self.masks_dir}")
        print(f"Phase: {phase}, Augmentations: {'ON' if phase == 'train' else 'OFF'}")
        
        self.samples = []
        mask_files = list(self.masks_dir.glob("*.png"))
        print(f"Found {len(mask_files)} mask files")
        
        for mask_path in sorted(mask_files):
            stem = mask_path.stem
            image_path = find_image_for_stem(self.images_dir, stem)
            if image_path is not None:
                self.samples.append((image_path, mask_path))
            else:
                print(f"Warning: No image found for mask {stem}")

        if not self.samples:
            raise RuntimeError(f"No paired image/mask samples found")

        print(f"Found {len(self.samples)} paired samples")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]
        
        # Чтение изображения
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise RuntimeError(f"Cannot read image: {image_path}")
        
        # Конвертация BGR -> RGB
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        
        # Чтение маски
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Cannot read mask: {mask_path}")
        
        # Бинаризация маски
        mask = (mask > 0).astype(np.float32)
        
        # Применяем трансформации
        image, mask = self.transform(image, mask)
        
        return image, mask

# =========================
# МОДЕЛЬ
# =========================

def build_model() -> nn.Module:
    if MODEL_NAME == "Unet":
        model = smp.Unet(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "FPN":
        model = smp.FPN(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")
    return model

# =========================
# ФУНКЦИИ ОБУЧЕНИЯ
# =========================

def train_one_epoch(model, loader, optimizer, loss_fn, device, scaler=None):
    model.train()
    
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    
    pbar = tqdm(loader, desc="Training", unit="batch")
    
    for batch_idx, (images, masks) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = loss_fn(logits, masks)
            
            scaler.scale(loss).backward()
            # Gradient clipping для стабильности
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = loss_fn(logits, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits.detach(), masks, threshold=THRESHOLD)
        running_iou += iou_score_from_logits(logits.detach(), masks, threshold=THRESHOLD)
        
        pbar.set_postfix({
            'loss': f'{running_loss/(batch_idx+1):.4f}',
            'dice': f'{running_dice/(batch_idx+1):.4f}',
        })
    
    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }

@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = loss_fn(logits, masks)

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits, masks, threshold=THRESHOLD)
        running_iou += iou_score_from_logits(logits, masks, threshold=THRESHOLD)

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }

# =========================
# КОНФИГУРАЦИЯ
# =========================

DATA_ROOT = Path(r"C:\Users\777\data\dl-lab-3-product-segmentation\train")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
SAVE_DIR = Path("./seg_train_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 512
BATCH_SIZE = 4
NUM_EPOCHS = 100
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.2
NUM_WORKERS = 0
SEED = 42
THRESHOLD = 0.5

MODEL_NAME = "UnetPlusPlus"
ENCODER_NAME = "efficientnet-b6"
ENCODER_WEIGHTS = "imagenet"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# ОСНОВНАЯ ФУНКЦИЯ
# =========================

def main():
    seed_everything(SEED)
    
    print("="*50)
    print("Step 1: Creating datasets...")
    print("="*50)
    
    # Создаем тренировочный и валидационный датасеты отдельно
    # Сначала получаем все индексы
    full_dataset = BinarySegDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        img_size=IMG_SIZE,
        phase='train'
    )
    
    print("\n" + "="*50)
    print("Step 2: Splitting dataset...")
    print("="*50)
    
    val_size = int(len(full_dataset) * VAL_RATIO)
    train_size = len(full_dataset) - val_size
    
    generator = torch.Generator().manual_seed(SEED)
    train_indices, val_indices = random_split(
        list(range(len(full_dataset))), 
        [train_size, val_size], 
        generator=generator
    )
    
    # Создаем отдельные датасеты с правильными трансформациями
    train_dataset = BinarySegDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        img_size=IMG_SIZE,
        phase='train'  # С аугментациями
    )
    
    val_dataset = BinarySegDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        img_size=IMG_SIZE,
        phase='val'    # Без аугментаций
    )
    
    # Используем Subset для выбора нужных индексов
    from torch.utils.data import Subset
    train_dataset = Subset(train_dataset, train_indices.indices)
    val_dataset = Subset(val_dataset, val_indices.indices)
    
    print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")
    
    print("\n" + "="*50)
    print("Step 3: Creating data loaders...")
    print("="*50)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE == 'cuda' else False,
        drop_last=False,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE == 'cuda' else False,
        drop_last=False,
    )
    
    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    
    # Тестируем загрузку одного батча
    print("\n" + "="*50)
    print("Step 4: Testing data loader...")
    print("="*50)
    try:
        test_batch = next(iter(train_loader))
        print(f"✓ Data loader works!")
        print(f"  Images shape: {test_batch[0].shape}")
        print(f"  Masks shape: {test_batch[1].shape}")
        print(f"  Image range: {test_batch[0].min():.2f} to {test_batch[0].max():.2f}")
        print(f"  Mask unique values: {torch.unique(test_batch[1])}")
    except Exception as e:
        print(f"✗ Data loader failed: {e}")
        return
    
    print("\n" + "="*50)
    print("Step 5: Building model...")
    print("="*50)
    
    model = build_model().to(DEVICE)
    print(f"✓ Model built with {sum(p.numel() for p in model.parameters()):,} parameters")
    print(f"  Model: {MODEL_NAME}")
    print(f"  Encoder: {ENCODER_NAME}")
    
    # Включаем градиентное чекпоинтинг для экономии памяти
    if hasattr(model.encoder, 'set_gradient_checkpointing'):
        model.encoder.set_gradient_checkpointing(True)
        print("✓ Gradient checkpointing enabled")
    
    # Mixed precision training
    scaler = torch.amp.GradScaler('cuda') if DEVICE == "cuda" else None
    if scaler:
        print("✓ AMP (mixed precision) enabled")
    
    # КОМБИНИРОВАННАЯ ФУНКЦИЯ ПОТЕРЬ
    loss_fn = CombinedLoss(dice_weight=0.5, bce_weight=0.3, focal_weight=0.2)
    print("✓ Combined Loss (Dice + BCE + Focal) enabled")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    
    print("\n" + "="*50)
    print("Step 6: Starting training...")
    print("="*50)
    
    best_val_dice = -1.0
    history = []
    
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch}/{NUM_EPOCHS}")
        print(f"{'='*50}")
        
        epoch_start = time.time()
        
        train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE, scaler)
        val_metrics = validate_one_epoch(model, val_loader, loss_fn, DEVICE)
        
        scheduler.step()

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_dice": train_metrics["dice"],
            "train_iou": train_metrics["iou"],
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
            "val_iou": val_metrics["iou"],
        }
        history.append(row)
        
        epoch_time = time.time() - epoch_start
        print(f"Epoch {epoch} completed in {epoch_time:.1f}s")
        print(f"  Train - Loss: {train_metrics['loss']:.4f}, Dice: {train_metrics['dice']:.4f}, IoU: {train_metrics['iou']:.4f}")
        print(f"  Val   - Loss: {val_metrics['loss']:.4f}, Dice: {val_metrics['dice']:.4f}, IoU: {val_metrics['iou']:.4f}")
        
        # Сохраняем последнюю модель
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": row["val_dice"],
                "config": {
                    "MODEL_NAME": MODEL_NAME,
                    "ENCODER_NAME": ENCODER_NAME,
                    "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                    "IMG_SIZE": IMG_SIZE,
                },
            },
            SAVE_DIR / "last.pth",
        )

        # Сохраняем лучшую модель
        if row["val_dice"] > best_val_dice:
            best_val_dice = row["val_dice"]
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_dice": row["val_dice"],
                    "config": {
                        "MODEL_NAME": MODEL_NAME,
                        "ENCODER_NAME": ENCODER_NAME,
                        "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                        "IMG_SIZE": IMG_SIZE,
                    },
                },
                SAVE_DIR / "best.pth",
            )
            print(f"✓ Saved new best model with val_dice={best_val_dice:.4f}")
        
        # Очищаем кэш GPU
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    print(f"\n{'='*50}")
    print(f"✓ Training finished!")
    print(f"Best val_dice: {best_val_dice:.4f}")
    print(f"Models saved to: {SAVE_DIR}")
    print(f"{'='*50}")

if __name__ == "__main__":
    main()

CUDA available: False
Step 1: Creating dataset...
Loading dataset from C:\Users\777\data\dl-lab-3-product-segmentation\train/images and C:\Users\777\data\dl-lab-3-product-segmentation\train/masks
Found 0 mask files


RuntimeError: No paired image/mask samples found

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn
import torchvision.transforms as T
from torchvision.transforms import functional as F

# =========================
# CONFIG
# =========================
TEST_IMAGES_DIR = Path(r"C:\Users\YourName\data\dl-lab-3-product-segmentation\test_images")  # ИЗМЕНИТЕ
OUTPUT_CSV = "submission.csv"
CHECKPOINT_PATH = Path(r"./seg_train_runs/best.pth")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# HELPERS
# =========================
def cv2_imread_unicode(path: Path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, flags)

def build_model(model_name: str, encoder_name: str, encoder_weights=None):
    if model_name == "Unet":
        model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif model_name == "FPN":
        model = smp.FPN(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {model_name}")
    return model

def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.astype(np.uint8).tolist(), separators=(",", ":"))

def preprocess_image(image: np.ndarray, img_size: int, preprocess_input=None) -> torch.Tensor:
    """Предобработка одного изображения"""
    # Ресайз
    image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = image.astype(np.float32)
    
    if preprocess_input is not None:
        image = preprocess_input(image)
    else:
        image = image / 255.0
    
    # Конвертация в тензор
    image = torch.from_numpy(image.transpose(2, 0, 1)).float()
    return image

@torch.no_grad()
def predict_with_tta(model, image_tensor, device, use_tta=True):
    """
    Предсказание с Test Time Augmentation
    """
    image_tensor = image_tensor.unsqueeze(0).to(device)
    
    if not use_tta:
        logits = model(image_tensor)
        return torch.sigmoid(logits)[0, 0]
    
    # Собираем предсказания с разными аугментациями
    predictions = []
    
    # 1. Оригинал
    logits = model(image_tensor)
    pred_orig = torch.sigmoid(logits)[0, 0]
    predictions.append(pred_orig)
    
    # 2. Горизонтальный флип
    flipped = torch.flip(image_tensor, dims=[-1])
    logits_flipped = model(flipped)
    pred_flipped = torch.flip(torch.sigmoid(logits_flipped)[0, 0], dims=[-1])
    predictions.append(pred_flipped)
    
    # 3. Вертикальный флип
    flipped_v = torch.flip(image_tensor, dims=[-2])
    logits_flipped_v = model(flipped_v)
    pred_flipped_v = torch.flip(torch.sigmoid(logits_flipped_v)[0, 0], dims=[-2])
    predictions.append(pred_flipped_v)
    
    # 4. Поворот на 90 градусов
    rotated_90 = torch.rot90(image_tensor, 1, dims=[-2, -1])
    logits_rot90 = model(rotated_90)
    pred_rot90 = torch.rot90(torch.sigmoid(logits_rot90)[0, 0], -1, dims=[-2, -1])
    predictions.append(pred_rot90)
    
    # 5. Поворот на 180 градусов
    rotated_180 = torch.rot90(image_tensor, 2, dims=[-2, -1])
    logits_rot180 = model(rotated_180)
    pred_rot180 = torch.rot90(torch.sigmoid(logits_rot180)[0, 0], -2, dims=[-2, -1])
    predictions.append(pred_rot180)
    
    # 6. Поворот на 270 градусов
    rotated_270 = torch.rot90(image_tensor, 3, dims=[-2, -1])
    logits_rot270 = model(rotated_270)
    pred_rot270 = torch.rot90(torch.sigmoid(logits_rot270)[0, 0], -3, dims=[-2, -1])
    predictions.append(pred_rot270)
    
    # Усредняем все предсказания
    pred_avg = torch.stack(predictions).mean(dim=0)
    
    return pred_avg

# =========================
# LOAD CHECKPOINT
# =========================
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

if "config" not in checkpoint:
    raise ValueError("Checkpoint does not contain 'config'.")

cfg = checkpoint["config"]

MODEL_NAME = cfg["MODEL_NAME"]
ENCODER_NAME = cfg["ENCODER_NAME"]
ENCODER_WEIGHTS = cfg.get("ENCODER_WEIGHTS", None)
IMG_SIZE = int(cfg["IMG_SIZE"])

print("="*50)
print("Loaded checkpoint config:")
print(f"MODEL_NAME     = {MODEL_NAME}")
print(f"ENCODER_NAME   = {ENCODER_NAME}")
print(f"ENCODER_WEIGHTS = {ENCODER_WEIGHTS}")
print(f"IMG_SIZE       = {IMG_SIZE}")
print("="*50)

# Загружаем модель
model = build_model(
    model_name=MODEL_NAME,
    encoder_name=ENCODER_NAME,
    encoder_weights=None,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()
print("✓ Model loaded successfully")

# Функция предобработки
preprocess_input = None
if ENCODER_WEIGHTS is not None:
    preprocess_input = get_preprocessing_fn(ENCODER_NAME, pretrained=ENCODER_WEIGHTS)

# =========================
# INFERENCE WITH TTA
# =========================
image_paths = sorted([p for p in TEST_IMAGES_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS])

if not image_paths:
    raise FileNotFoundError(f"No images found in: {TEST_IMAGES_DIR}")

print(f"\nFound {len(image_paths)} test images")
print("Using TTA (Test Time Augmentation) with 6 augmentations...")

rows = []

with torch.no_grad():
    for i, img_path in enumerate(image_paths, 1):
        # Чтение изображения
        img_bgr = cv2_imread_unicode(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            print(f"[skip] cannot read: {img_path}")
            continue

        H, W = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Предобработка
        img_tensor = preprocess_image(img_rgb, IMG_SIZE, preprocess_input)
        
        # Предсказание с TTA
        pred = predict_with_tta(model, img_tensor, DEVICE, use_tta=True)
        
        # Конвертация в numpy
        pred = pred.detach().cpu().numpy()
        
        # Возвращаем в исходный размер
        if pred.shape != (H, W):
            pred = cv2.resize(pred.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
        
        # Бинаризация
        mask = (pred > THRESHOLD).astype(np.uint8)
        
        # Пост-процессинг: удаляем маленькие компоненты
        if mask.sum() > 0:
            num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
            for j in range(1, num_labels):
                if stats[j, cv2.CC_STAT_AREA] < 100:  # Удаляем компоненты меньше 100 пикселей
                    mask[labels == j] = 0

        rows.append({
            "ImageId": img_path.name,
            "mask": serialize_mask(mask),
        })

        if i % 100 == 0 or i == len(image_paths):
            print(f"Processed {i}/{len(image_paths)}")

# Сохраняем результат
submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print("\n" + "="*50)
print(f"✓ Done! Saved submission to: {OUTPUT_CSV}")
print(f"Total processed: {len(rows)} images")
print(f"Sample of submission:")
print(submission_df.head())
print("="*50)